In [27]:
import pandas as pd
path="/content/spam.csv"
df=pd.read_csv(path, encoding='latin1')
df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)
df.columns = ['label', 'text']
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
import re

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')
!pip install gensim
from gensim.models import Word2Vec

df.info()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   5572 non-null   object
 1   text    5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [28]:
print(df['label'].value_counts())

label
ham     4825
spam     747
Name: count, dtype: int64


In [29]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase and remove punctuation/special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords, then Stem and Lemmatize
    processed_tokens = [lemmatizer.lemmatize(stemmer.stem(w)) for w in tokens if w not in stop_words]
    return processed_tokens

# Apply preprocessing
df['processed_text'] = df['text'].apply(preprocess_text)
display(df[['label','text', 'processed_text']].head(20))

,label,text,processed_text
0,ham,"Go until jurong point, crazy.. Available only ...","[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,Ok lar... Joking wif u oni...,"[ok, lar, joke, wif, u, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entri, wkli, comp, win, fa, cup, final,..."
3,ham,U dun say so early hor... U c already then say...,"[u, dun, say, earli, hor, u, c, alreadi, say]"
4,ham,"Nah I don't think he goes to usf, he lives aro...","[nah, dont, think, goe, usf, live, around, tho..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...,"[freemsg, hey, darl, week, word, back, id, lik..."
6,ham,Even my brother is not like to speak with me. ...,"[even, brother, like, speak, treat, like, aid,..."
7,ham,As per your request 'Melle Melle (Oru Minnamin...,"[per, request, mell, mell, oru, minnaminungint..."
8,spam,WINNER!! As a valued network customer you have...,"[winner, valu, network, custom, select, receiv..."
9,spam,Had your mobile 11 months or more? U R entitle...,"[mobil, month, u, r, entitl, updat, latest, co..."


In [30]:
# Train Word2Vec model
print("Training Word2Vec model...")
w2v_model = Word2Vec(sentences=df['processed_text'], vector_size=100, window=5, min_count=2, workers=4)

# Example: Find most similar words to a common term in the dataset
try:
    word = 'free'
    if word in w2v_model.wv:
        print(f"Words most similar to '{word}':")
        display(w2v_model.wv.most_similar(word))
except Exception as e:
    print(f"Could not find similar words: {e}")

print(f"Vocabulary size: {len(w2v_model.wv)}")

Training Word2Vec model...
Words most similar to 'free':


[('mobil', 0.9996210932731628),
 ('text', 0.9995960593223572),
 ('repli', 0.9995787143707275),
 ('tone', 0.9995735287666321),
 ('nokia', 0.9995460510253906),
 ('phone', 0.9995239973068237),
 ('txt', 0.9995172619819641),
 ('week', 0.9994688034057617),
 ('call', 0.9994438886642456),
 ('messag', 0.9994374513626099)]

Vocabulary size: 3337


In [31]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

# Calculate the ratio for scale_pos_weight (ham / spam)
counts = df['label'].value_counts()
ratio = counts['ham'] / counts['spam']
print(f"Calculated scale_pos_weight ratio: {ratio:.2f}")

# Function to average word vectors for a document
def get_document_vector(tokens, model, vector_size):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

# 1. Create feature matrix X
vector_size = w2v_model.vector_size
X = np.array([get_document_vector(tokens, w2v_model, vector_size) for tokens in df['processed_text']])

# 2. Encode labels (ham=0, spam=1)
le = LabelEncoder()
y = le.fit_transform(df['label'])

# 3. Train-Test Split with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Initialize and train XGBoost with scale_pos_weight
xgb_model = XGBClassifier(eval_metric='logloss', scale_pos_weight=ratio, random_state=42)
xgb_model.fit(X_train, y_train)

# 5. Evaluate the model
y_pred = xgb_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report (With scale_pos_weight):")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Calculated scale_pos_weight ratio: 6.46
Accuracy: 0.9587

Classification Report (With scale_pos_weight):
              precision    recall  f1-score   support

         ham       0.98      0.98      0.98       966
        spam       0.85      0.85      0.85       149

    accuracy                           0.96      1115
   macro avg       0.91      0.91      0.91      1115
weighted avg       0.96      0.96      0.96      1115



In [32]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid including scale_pos_weight tuning
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.1, 0.2],
    'scale_pos_weight': [1, ratio, ratio * 1.5]
}

# Initialize GridSearchCV focusing on F1 score
grid_search = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1
)

# Fit the grid search
print("Starting Grid Search with weighted classes...")
grid_search.fit(X_train, y_train)

# Best parameters and score
print(f"Best Parameters: {grid_search.best_params_}")

# Evaluate the best model
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("\nOptimized Model Classification Report (After Weighted GridSearch):")
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

Starting Grid Search with weighted classes...
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best Parameters: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'scale_pos_weight': np.float64(6.459170013386881)}

Optimized Model Classification Report (After Weighted GridSearch):
              precision    recall  f1-score   support

         ham       0.98      0.98      0.98       966
        spam       0.85      0.85      0.85       149

    accuracy                           0.96      1115
   macro avg       0.91      0.91      0.91      1115
weighted avg       0.96      0.96      0.96      1115



In [33]:
def predict_message(message):
    # 1. Preprocess the input text
    processed_input = preprocess_text(message)

    # 2. Convert to document vector
    vector_size = w2v_model.vector_size
    doc_vector = get_document_vector(processed_input, w2v_model, vector_size)
    doc_vector = doc_vector.reshape(1, -1) # Reshape for a single prediction

    # 3. Predict using the optimized model
    prediction_idx = best_model.predict(doc_vector)[0]
    prediction_label = le.inverse_transform([prediction_idx])[0]

    # 4. Get probability
    probabilities = best_model.predict_proba(doc_vector)[0]
    confidence = probabilities[prediction_idx]

    return prediction_label, confidence

# --- Test it here ---
test_message = "i was with my boyfriend why are u getting angry,we are friends right" # @param {type:"string"}

label, conf = predict_message(test_message)
print(f"Message: {test_message}")
print(f"Prediction: {label.upper()}")
print(f"Confidence: {conf:.2%}")

Message: i was with my boyfriend why are u getting angry,we are friends right
Prediction: HAM
Confidence: 99.67%
